In [5]:
import os
import PyPDF2
import json
import re


from nltk.stem import PorterStemmer
from nltk.stem import LancasterStemmer
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

In [9]:


# Initialize the stemmer
stemmer = SnowballStemmer(language='portuguese')

def clean_and_stem_text(text):
    # Remove new lines and words smaller than 3 letters
    text = re.sub(r'\b(?!\d+)\w{1,2}\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize the text
    words = word_tokenize(text)

    # Apply stemming
    stemmed_words = [stemmer.stem(word) for word in words]

    # Join the stemmed words back into a single string
    cleaned_text = ' '.join(stemmed_words)

    return cleaned_text

def pdf_to_json(pdf_path):
    with open(pdf_path, 'rb') as pdf_file:
        read_pdf = PyPDF2.PdfReader(pdf_file)
        number_of_pages = len(read_pdf.pages)
        pdf_text = []

        for i in range(number_of_pages):
            page = read_pdf.pages[i]
            extracted_text = page.extract_text()
            if extracted_text:
                cleaned_text = clean_and_stem_text(extracted_text)
                pdf_text.append(cleaned_text)

    pdf_name = os.path.splitext(os.path.basename(pdf_path))[0]
    pdf_data = {"name": pdf_name, "content": " ".join(pdf_text)}  # Ensure content is a single string

    return pdf_data

def convert_pdfs_in_directory(input_directory, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for filename in os.listdir(input_directory):
        if filename.endswith(".pdf"):
            input_path = os.path.join(input_directory, filename)
            pdf_data = pdf_to_json(input_path)

            output_path = os.path.join(output_directory, f"{os.path.splitext(filename)[0]}.json")
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(pdf_data, f, ensure_ascii=False, indent=4)

# Define input and output directories
input_directory = "../bulas_pdf"
output_directory = "../bulas_json"
convert_pdfs_in_directory(input_directory, output_directory)
